# 03d · M/M/1 대기행렬과 리틀의 법칙 (The M/M/1 Queue and Little's Law)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.5 (마르코프 대기행렬), §3.2 (리틀의 공식, PASTA) · Ross 12e §8.2 (비용 항등식·리틀의 공식), §8.3.1 (단일 서버 지수 모형) · Norris §5.2.1 (M/M/1) · G&S 4e §11.2 (M/M/1) |
| 선수 노트북 | 03c (출생-사망 과정의 정상분포), 02d (푸아송 과정의 검사 역설·잔여시간) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **대기행렬은 출생-사망 CTMC이고, 그 정상분포 하나에서 '평균 몇 명이 있는가($L$)'와 '평균 얼마나 머무는가($W$)'가 모두 나온다.** 둘을 잇는 것이 리틀의 법칙(Little's law) $L=\lambda W$ — 분포와 무관한 회계 항등식이다. 시뮬레이션은 이제 상태가 아니라 **손님** 단위(Lindley 재귀)로 하고, 손님들의 체류시간이 강하게 자기상관된다는 것이 이 노트북의 통계적 교훈이다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import mmc_metrics, mm1_stationary
from spkit.plots import plot_hist_vs_pdf, plot_hist_vs_pmf

SEED, rng = rng_for("03d")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (이 노트북에서는 손님 수의 척도로만 참고)
N_CUST = scale(200_000)   # 손님 단위 시뮬레이션의 손님 수 (FAST: 20_000)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_CUST={N_CUST}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03c) 출생-사망 과정의 정상분포 곱 형태와 그 유도의 핵심(절단 방정식)은?

<details><summary>정답</summary>

$\pi_n=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$. 핵심은 절단 방정식 $\lambda_n\pi_n=\mu_{n+1}\pi_{n+1}$ — 상태 $n$과 $n+1$ 사이의 '벽'을 넘는 양방향 흐름이 정상 상태에서 같다는 것.

</details>

**Q2.** (03c) M/M/∞의 정상분포와, 빈 상태에서 출발한 $X(t)$의 분포는?

<details><summary>정답</summary>

정상분포는 $\mathrm{Poisson}(\lambda/\mu)$; 시각 $t$의 분포는 $\mathrm{Poisson}\!\big(\tfrac\lambda\mu(1-e^{-\mu t})\big)$ (푸아송 세분화: 시각 $t$에 아직 남아 있을 확률로 도착을 솎아낸다).

</details>

**Q3.** (03c) 점프 사슬의 정상분포 $\nu$와 CTMC의 정상분포 $\pi$의 관계는?

<details><summary>정답</summary>

$\pi_i\propto\nu_i/q_i$ — 방문 횟수(점프 사슬)에 평균 체류시간 $1/q_i$를 곱한다.

</details>

**Q4.** (03b) $P(t)=e^{Qt}$를 계산할 때 `np.exp(Q*t)`를 쓰면 안 되는 이유는?

<details><summary>정답</summary>

원소별 지수는 행렬 지수 $\sum_n(Qt)^n/n!$과 전혀 다르다(행합이 1이 되지도 않는다). `scipy.linalg.expm`, 즉 spkit의 `transition_matrix(Q, t)`를 써야 한다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **M/M/1** (Kendall 표기 도착/서비스/서버 수): 비율 $\lambda$의 푸아송 도착, i.i.d. $\mathrm{Exp}(\mu)$ 서비스, 서버 1대, FIFO(first in, first out), 무한 대기실. $N(t)$ = 시각 $t$의 시스템 내 손님 수(서비스 중 + 대기 중)는 $\lambda_n=\lambda$, $\mu_n=\mu$인 출생-사망 CTMC다. **이용률(utilization)** $\rho=\lambda/\mu$.
- **정상분포**: $\rho<1$일 때 $\pi_n=(1-\rho)\rho^n$ (기하). 따라서 $P(\text{서버 바쁨})=1-\pi_0=\rho$ — 이용률이라는 이름의 이유.
- **시간평균 지표** (누가 언제 봐도 같은 '재고'): $L=E_\pi[N]=\rho/(1-\rho)$ (시스템 내 평균 손님 수), $L_q=E_\pi[(N-1)^+]=L-\rho=\rho^2/(1-\rho)$ (대기열 평균 길이).
- **손님평균 지표** (손님 한 명이 겪는 것): $W$ = 평균 체류시간(sojourn time; 도착~퇴장), $W_q$ = 평균 대기시간(waiting time; 도착~서비스 시작). 항상 $W=W_q+1/\mu$. M/M/1: $W=1/(\mu-\lambda)$, $W_q=\rho/(\mu-\lambda)$.
- **리틀의 법칙(Little's law)**: $L=\lambda W$, $L_q=\lambda W_q$, 그리고 (서비스 중인 평균 손님 수) $=\lambda\cdot\frac1\mu=\rho$. 분포·서비스 규칙에 무관한 회계 항등식이다.
- **PASTA** (Poisson Arrivals See Time Averages): 도착 직전 상태 $N(A_i^-)$의 장기 경험분포 = 시간평균 분포 $\pi$. 특히 $P(\text{도착 손님이 기다리지 않음})=\pi_0=1-\rho$.
- **Lindley 재귀** (손님 단위 시뮬레이션): 도착 $A_i=A_{i-1}+\mathrm{Exp}(\lambda)$, 서비스 $S_i\sim\mathrm{Exp}(\mu)$, 퇴장 $D_i=\max(A_i,D_{i-1})+S_i$ (서비스는 '내가 도착한 뒤, 앞 손님이 떠난 뒤'에 시작), 체류 $W_i=D_i-A_i$, 대기 $W_{q,i}=\max(A_i,D_{i-1})-A_i$.
- **배치 평균(batch means) SE**: 자기상관 있는 수열 $x_1,\dots,x_n$을 $b$개의 연속 블록으로 나눠 블록 평균들의 표준편차$/\sqrt b$. 이 노트북에서는 $b=20$, 워밍업으로 처음 10%를 버린다.
- **이 노트북의 기본 수치**: $\lambda=0.8,\ \mu=1$ ($\rho=0.8$). 정리 1·4에서 $L=4,\ L_q=3.2,\ W=5,\ W_q=4$.

### 2.2 정리 1 — M/M/1 정상분포와 $L$ (Durrett 3e §4.5, Ross 12e §8.3.1, Norris §5.2.1)

> $\rho=\lambda/\mu<1$이면 $\pi_n=(1-\rho)\rho^n$, $L=\rho/(1-\rho)$, $L_q=\rho^2/(1-\rho)$. $\rho\ge1$이면 정상분포가 없다.

**가정이 왜 필요한가**
- **$\rho<1$**: 곱 형태 $\pi_n\propto\rho^n$의 합 $\sum\rho^n$이 유한하려면 필요하다(03c 정리 2의 정규화 조건). $\rho\ge1$이면 대기열이 발산한다 — 들어오는 속도가 처리 속도 이상이면 줄이 끝없이 길어진다.
- **푸아송 도착·지수 서비스**: 둘 다 무기억이어야 $N(t)$ 하나만으로 마르코프(출생-사망)가 된다. 일반 서비스 분포(M/G/1)에서는 '현재 서비스가 얼마나 진행됐는가'를 함께 기억해야 하고, 그러면 $N(t)$만으로는 마르코프가 아니다.

<details><summary>증명</summary>

03c 정리 2(출생-사망 정상분포)에서 $\pi_n=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$이고, 여기서는 $\lambda_k/\mu_{k+1}=\lambda/\mu=\rho$가 상수이므로 $\pi_n=\pi_0\rho^n$. $\rho<1$이면 $\sum_n\pi_n=\pi_0/(1-\rho)=1$에서 $\pi_0=1-\rho$. ($\rho\ge1$이면 $\sum\rho^n=\infty$라 정규화가 불가능하고, 이는 03c의 기준에 따라 사슬이 양재귀가 아님을 뜻한다.)

$L$은 기하급수를 미분해서 얻는다: $\sum_{n\ge0}n\rho^n=\rho\frac{d}{d\rho}\sum_n\rho^n=\rho\frac{d}{d\rho}\frac1{1-\rho}=\frac{\rho}{(1-\rho)^2}$이므로
$$L=\sum_n n(1-\rho)\rho^n=(1-\rho)\cdot\frac{\rho}{(1-\rho)^2}=\frac{\rho}{1-\rho}.$$

$L_q$는 서비스 중인 한 명을 뺀 것: $L_q=\sum_{n\ge1}(n-1)\pi_n=\sum_{n\ge1}n\pi_n-\sum_{n\ge1}\pi_n=L-(1-\pi_0)=L-\rho=\frac{\rho}{1-\rho}-\rho=\frac{\rho^2}{1-\rho}$. $\square$

</details>

### 2.3 정리 2 — 리틀의 법칙, 표본경로 버전 (Durrett 3e §3.2, Ross 12e §8.2)

> 시각 0에 시스템이 비어 있고 $n$명의 손님이 모두 시각 $T$ 이전에 떠나 $T$에 다시 비어 있으면
> $$\int_0^T N(t)\,dt=\sum_{i=1}^n W_i$$
> 가 **정확히** 성립한다. 따라서 $\bar L(T)=\hat\lambda(T)\,\bar W$ ($\hat\lambda=n/T$, $\bar W=\frac1n\sum_iW_i$). $T\to\infty$에서 극한들이 존재하면 $L=\lambda W$.

**가정이 왜 필요한가**
- **시작과 끝이 비어 있음**: 아니면 경계 손님의 체류시간 일부가 $[0,T]$ 밖에 있어 항등식에 경계항이 붙는다. 극한에서는 이 경계항이 $T$에 비해 사라진다(Stidham 1974) — 그래서 일반 버전은 극한 정리다.
- **극한의 존재($\rho<1$)**: 안정성이 없으면 $L=W=\infty$로 식이 무의미해진다.
- **분포·서비스 규칙에 대한 가정은 없다**: FIFO가 아니어도, 푸아송이 아니어도, 서버가 몇 대여도 성립한다(E3에서 D/M/1로 확인). 이것이 리틀의 법칙이 강력한 이유다.

<details><summary>증명</summary>

손님 $i$가 시스템에 있는 시간 구간은 $[A_i,D_i)$이므로 $N(t)=\sum_{i=1}^n\mathbf 1\{A_i\le t<D_i\}$. 가정에 의해 모든 구간 $[A_i,D_i)\subset[0,T]$. 양변을 $[0,T]$에서 적분하고 유한합과 적분을 교환하면
$$\int_0^TN(t)\,dt=\sum_{i=1}^n\int_0^T\mathbf 1\{A_i\le t<D_i\}\,dt=\sum_{i=1}^n(D_i-A_i)=\sum_{i=1}^nW_i.$$
그림으로: $N(t)$ 계단함수 아래의 면적은 '단위 높이의 가로 막대' $[A_i,D_i)\times[0,1)$ $n$개를 쌓은 것이다. 면적을 **세로로**(시각별 손님 수를 적분) 세나 **가로로**(손님별 막대 길이 = 체류시간을 합산) 세나 같다. 양변을 $T$로 나누면 $\bar L(T)=\frac1T\int_0^TN=\frac nT\cdot\frac1n\sum_iW_i=\hat\lambda(T)\bar W$.

일반 버전(Stidham 1974; Ross 12e §8.2의 비용 항등식): 시각 $T$에 남아 있는 손님의 체류시간은 일부만 $[0,T]$ 안에 들어오므로 $\sum_{i:A_i\le T}W_i\ge\int_0^TN\ge\sum_{i:D_i\le T}W_i$. 양쪽을 $T$로 나누고 $\hat\lambda(T)\to\lambda$, $\bar W\to W$ (유한)를 쓰면 두 경계가 모두 $\lambda W$로 수렴해 $L=\lambda W$. $\square$

</details>

### 2.4 정리 3 — PASTA (Durrett 3e §3.2; Wolff 1982)

> 도착이 푸아송이고 시스템 상태가 미래 도착에 의존하지 않으면(lack of anticipation), 도착 직전 상태의 장기 경험분포는 시간평균 분포와 같다. M/M/1에서 도착이 보는 $N$은 기하분포 $(1-\rho)\rho^n$, 특히 $P(W_q=0)=1-\rho$.

**가정이 왜 필요한가**
- **푸아송(독립 증분)**: $[t,t+h)$에 도착이 있을 확률 $\lambda h$가 시각 $t$의 상태와 독립이어야 '도착 시각에서 본 상태'가 '아무 시각에서 본 상태'와 같다. 결정론적 도착(D/M/1)에서는 도착이 규칙적으로 배치되어 앞 손님이 떠날 시간을 '벌어 주는' 셈이라 도착이 보는 빈 시스템 비율이 시간평균보다 크다(E3).
- **lack of anticipation**: 서비스가 미래 도착 시각을 보고 결정되면(예: 다음 손님이 오기 직전에 서비스를 끝내는 서버) 도착이 보는 상태가 편향된다.

증명은 Wolff (1982)와 Durrett 3e §3.2를 인용한다(마팅게일 논증). 직관: 푸아송 과정은 '미래를 훔쳐보지 않는' 균일한 관찰자라, 도착 시각들이 시간축의 무작위 표본과 같다. 이 노트북에서는 도착이 본 $N$의 경험 pmf를 $\pi$와 겹쳐 확인한다.

### 2.5 정리 4 — FIFO M/M/1의 체류시간은 $\mathrm{Exp}(\mu-\lambda)$ (Ross 12e §8.3.1, G&S 4e §11.2)

> 정상 상태의 FIFO M/M/1에서 손님의 체류시간 $W\sim\mathrm{Exp}(\mu-\lambda)$. 따라서 $E[W]=1/(\mu-\lambda)$, $P(W>t)=e^{-(\mu-\lambda)t}$. 대기시간은 $P(W_q>t)=\rho\,e^{-(\mu-\lambda)t}$ (0에 질량 $1-\rho$의 원자).

**가정이 왜 필요한가**
- **PASTA**: 도착이 보는 $N$이 $\pi$(기하)여야 한다.
- **지수 서비스의 무기억성**: 서비스 중인 손님의 *잔여* 서비스도 $\mathrm{Exp}(\mu)$여야 체류시간이 '$N+1$개의 i.i.d. $\mathrm{Exp}(\mu)$ 합'이 된다.
- **FIFO**: 다른 규칙(LIFO, 처리공유)에서는 리틀의 법칙 때문에 평균 $W$는 같지만 **분포**가 다르다.

<details><summary>증명</summary>

정리 3에 의해 도착 손님이 앞에 $N=n$명을 볼 확률은 $(1-\rho)\rho^n$. FIFO이므로 이 손님의 체류시간은 (앞 $n$명의 서비스 — 서비스 중인 사람의 잔여 서비스는 무기억성으로 $\mathrm{Exp}(\mu)$) + (자기 서비스) = $n+1$개 i.i.d. $\mathrm{Exp}(\mu)$의 합, 즉 $\mathrm{Erlang}(n+1,\mu)$. 라플라스 변환으로 섞으면
$$E[e^{-sW}]=\sum_{n\ge0}(1-\rho)\rho^n\Big(\frac{\mu}{\mu+s}\Big)^{n+1}=\frac{(1-\rho)\mu}{\mu+s}\cdot\frac{1}{1-\rho\mu/(\mu+s)}=\frac{(1-\rho)\mu}{\mu+s-\rho\mu}=\frac{\mu-\lambda}{\mu-\lambda+s},$$
이는 $\mathrm{Exp}(\mu-\lambda)$의 변환이다. (02a의 '기하 개수의 지수 합은 지수' — 비율 $\mu$의 푸아송 과정에서 각 점을 확률 $1-\rho$로 표시하면 첫 표시점까지의 시간은 $\mathrm{Exp}((1-\rho)\mu)=\mathrm{Exp}(\mu-\lambda)$ — 로도 같은 결론.)

대기시간: $n=0$이면 $W_q=0$ (확률 $1-\rho$), $n\ge1$이면 $n$개의 합 $\mathrm{Erlang}(n,\mu)$. 조건부 변환은 $\sum_{n\ge1}(1-\rho)\rho^n(\mu/(\mu+s))^n\big/\rho=\frac{\mu-\lambda}{\mu-\lambda+s}$이므로 $W_q\mid W_q>0\sim\mathrm{Exp}(\mu-\lambda)$, 따라서 $P(W_q>t)=\rho\,e^{-(\mu-\lambda)t}$. $\square$

</details>

### 2.6 직관

$L=\rho/(1-\rho)$: $\rho=0.5$면 1명, $0.8$이면 4명, $0.9$면 9명, $0.99$면 99명 — 이용률을 조금 올리면 대기가 **폭발**한다. 서버를 '효율적으로' 90% 이상 돌리려는 순간 손님이 겪는 대기가 걷잡을 수 없이 커진다는 것이 대기행렬 이론의 첫 번째 경영 교훈이다.

리틀의 법칙은 '재고 = 유입률 × 체류시간'이라는 회계 항등식이다. 그림으로는 $N(t)$ 계단함수 아래 면적을 가로로 세면 체류시간의 합, 세로로 세면 손님 수의 적분이라는 것뿐이다. 그래서 시뮬레이션에서 $\hat L=\hat\lambda\bar W$는 항등식이라 **자동으로** 맞는다 — 진짜 검증은 $\bar W$ vs $1/(\mu-\lambda)$, $\hat L$ vs $\rho/(1-\rho)$ 쪽이다.

PASTA는 '푸아송은 미래를 훔쳐보지 않는다': 도착 시각이 상태와 무관한 무작위 표본이므로 도착이 보는 것이 곧 시간평균이다. 체류시간이 정확히 지수인 것은 '기하 개수의 지수 합은 지수'라는 02a의 사실이 다시 나타난 것이다.

### 2.7 함정(traps)

1. **$W$(체류)와 $W_q$(대기) 혼동**: $W=W_q+1/\mu$. 그리고 $L_q=L-\rho$이지 $L-1$이 아니다 — 서버가 놀고 있을 때는 1을 빼면 안 된다.
2. **손님 체류시간은 강하게 자기상관**된다: $\rho=0.8$에서 유효 표본 수가 약 $1/100$. i.i.d. 공식 `std/sqrt(n)`은 SE를 약 10배 과소평가한다. 배치 평균이나 독립 반복을 써야 한다(4.8에서 비율을 출력).
3. **리틀의 법칙의 시뮬레이션 '검증'은 항상 맞는다** — 표본경로 항등식이기 때문이다. 검증할 것은 닫힌 형식 $L=\rho/(1-\rho)$, $W=1/(\mu-\lambda)$ 쪽이다.
4. **PASTA는 푸아송 도착에만** 성립한다. D/M/1에서는 도착이 보는 빈 확률이 $1-\rho$보다 크다(E3: 약 0.37 vs 0.2).
5. **워밍업**: 빈 상태에서 시작하면 초기 과도가 있다. 완화 시간은 대략 $1/(\sqrt\mu-\sqrt\lambda)^2$ ($\rho=0.8$: 약 90 시간 단위 ≈ 72명). 처음 10%를 버리면 충분하지만 $\rho\to1$에서는 급증한다(4.7).
6. **`rng.exponential(1/lam)`**: 인자는 비율이 아니라 **평균**이다(03a 트랩 재확인).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요. (a) 리틀의 법칙을 계단함수 그림 하나로 설명하라. (b) 왜 $\bar W$의 표준오차를 `std/sqrt(n)`으로 계산하면 안 되는지 한 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 기본 수치는 $\lambda=0.8$, $\mu=1$ ($\rho=0.8$).

In [ ]:
predictions = {
    # lambda=0.8, mu=1 인 M/M/1 의 장기 평균 시스템 내 손님 수 L 은?
    "L_mm1": None,
    # 손님의 평균 체류시간 W (도착~퇴장) 는?
    "W_mm1": None,
    # 손님의 평균 대기시간 W_q (도착~서비스 시작) 는?
    "Wq_mm1": None,
    # 도착한 손님이 기다리지 않고 바로 서비스받을 확률은?
    "p_no_wait": None,
    # 손님의 체류시간이 10 을 넘을 확률은?
    "p_sojourn_gt_10": None,
    # 이용률을 rho = 0.95 로 올리면 L 은?
    "L_rho_095": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 손님 단위 시뮬레이션 도구

이 노트북의 시뮬레이션은 03a의 질레스피(상태 단위)가 아니라 **손님 단위**다: 손님 $i$의 도착·퇴장 시각을 Lindley 재귀로 만들면 체류시간·대기시간이 손님별로 나오고, $N(t)$는 도착·퇴장 사건을 병합해 복원한다. 자기상관 있는 수열의 SE를 위한 배치 평균도 여기서 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def mm1_customers(lam, mu, n_customers, rng):
    """FIFO M/M/1 손님 단위 시뮬레이션 (Lindley 재귀). returns (A, D): 도착·퇴장 시각."""
    A = np.cumsum(rng.exponential(1.0 / lam, n_customers))   # 인자는 평균 1/lam (트랩 6)
    S = rng.exponential(1.0 / mu, n_customers)
    D = np.empty(n_customers); prev = 0.0
    for i in range(n_customers):
        prev = max(A[i], prev) + S[i]      # 서비스 시작 = max(도착, 앞 손님 퇴장)
        D[i] = prev
    return A, D

def batch_means_se(x, n_batches=20):
    """자기상관 있는 수열의 평균에 대한 SE (배치 평균)."""
    x = np.asarray(x, float); m = len(x) // n_batches
    b = x[: m * n_batches].reshape(n_batches, m).mean(axis=1)
    return b.std(ddof=1) / np.sqrt(n_batches)

def number_in_system(A, D):
    """도착·퇴장 시각으로 N(t) 계단함수 (t_events, N) 생성."""
    t = np.concatenate([A, D]); d = np.concatenate([np.ones(len(A)), -np.ones(len(D))])
    o = np.argsort(t, kind="stable")
    return t[o], np.cumsum(d[o])

### 4.2 표본경로: $N(t)$ 계단함수 (단계 3a)

$\lambda=0.8,\mu=1$로 `N_CUST`명을 시뮬레이션하고, 처음 200 시간 단위의 $N(t)$를 그린다. 회색 면적이 정리 2의 $\int N(t)\,dt$ — 가로로 세면 체류시간의 합이다.

In [ ]:
LAM, MU = 0.8, 1.0
RHO = LAM / MU
n = N_CUST
A, D = mm1_customers(LAM, MU, n, rng)
start = np.maximum(A, np.concatenate([[0.0], D[:-1]]))   # 서비스 시작 시각
W, Wq = D - A, start - A                                  # 체류시간, 대기시간
t_ev, N = number_in_system(A, D)
print(f"n = {n} customers, last departure T = {D[-1]:.1f}, max N(t) = {N.max()}")

T_SHOW = 200.0
k = np.searchsorted(t_ev, T_SHOW)
fig, ax = plt.subplots()
ax.step(t_ev[:k], N[:k], where="post", lw=1)
ax.fill_between(t_ev[:k], N[:k], step="post", alpha=0.2, label="area = integral of N(t)")
ax.axhline(RHO / (1 - RHO), color="C3", ls="--", label="L = rho/(1-rho) = 4")
ax.set_xlabel("t"); ax.set_ylabel("N(t): customers in system")
ax.set_title("Fig 1: Number in system N(t) for the first 200 time units, rho = 0.8")
ax.legend()
plt.show()

빈 상태에서 출발하므로 처음 얼마간은 $L=4$보다 낮게 머무는 초기 과도(트랩 5)가 보일 수 있다. 200 시간 단위 안에 도착이 약 160명이라 $N(t)$가 0으로 돌아오는 '빈 구간'도 여러 번 나타난다 — 시간의 $1-\rho=20\%$는 서버가 논다.

### 4.3 닫힌 형식과 $\rho\to1$ 폭발 곡선 (단계 1)

`mmc_metrics(λ, μ, c=1)`은 M/M/c 일반 공식(Erlang C, 03e)의 $c=1$ 특수경우다. 정리 1·4의 값과 부동소수점 수준에서 일치해야 하고, 절단 정상분포 `mm1_stationary`의 $\sum n\pi_n$도 $L$과 같아야 한다.

In [ ]:
m = mmc_metrics(LAM, MU, 1)
L_exact, Lq_exact = RHO / (1 - RHO), RHO**2 / (1 - RHO)          # 정리 1
W_exact, Wq_exact = 1 / (MU - LAM), RHO / (MU - LAM)              # 정리 4 (또는 리틀)
np.testing.assert_allclose([m["L"], m["W"], m["Lq"], m["Wq"], m["p_wait"]],
                           [L_exact, W_exact, Lq_exact, Wq_exact, RHO], rtol=1e-10)
np.testing.assert_allclose([LAM * W_exact, LAM * Wq_exact], [L_exact, Lq_exact])   # 리틀 L = lam W
pi = mm1_stationary(LAM, MU, n_max=200)
L_from_pi = np.sum(np.arange(201) * pi)
print("mmc_metrics(0.8, 1, 1) =", {k: round(v, 6) for k, v in m.items()})
print(f"sum n*pi_n (n_max=200) = {L_from_pi:.15f}, |error| = {abs(L_from_pi - L_exact):.1e}")

In [ ]:
rho_grid = np.linspace(0.02, 0.98, 200)
fig, ax = plt.subplots()
ax.plot(rho_grid, rho_grid / (1 - rho_grid), label="L = rho / (1 - rho)")
for r in (0.5, 0.8, 0.9, 0.95):
    ax.plot(r, r / (1 - r), "o", color="C3")
    ax.annotate(f"rho={r}: L={r/(1-r):g}", (r, r / (1 - r)), xytext=(-70, 8), textcoords="offset points")
ax.set_xlabel("utilization rho = lambda / mu"); ax.set_ylabel("L (mean number in system)")
ax.set_title("Fig 2: L = rho/(1-rho): the blow-up as rho -> 1")
ax.legend()
plt.show()

### 4.4 체류·대기시간 측정과 $\mathrm{Exp}(\mu-\lambda)$ (단계 2)

처음 10%를 워밍업으로 버리고, 나머지 손님의 체류시간 $\bar W$, 대기시간 $\bar W_q$, 기다리지 않은 비율, $P(W>10)$을 잰다. 모든 SE는 **배치 평균**(20개 블록)이다.

In [ ]:
warm = n // 10
Ww, Wqw = W[warm:], Wq[warm:]
Wbar, se_W = Ww.mean(), batch_means_se(Ww, 20)
Wq_bar, se_Wq = Wqw.mean(), batch_means_se(Wqw, 20)
p0_hat, se_p0 = (Wqw == 0).mean(), batch_means_se((Wqw == 0).astype(float), 20)
p10_hat, se_p10 = (Ww > 10).mean(), batch_means_se((Ww > 10).astype(float), 20)
p10_exact = np.exp(-(MU - LAM) * 10)                            # 정리 4
print(f"after warm-up of {warm} customers ({n - warm} used):")
print(f"  W bar   = {Wbar:.3f} ± {se_W:.3f}   (exact {W_exact:.4f})")
print(f"  Wq bar  = {Wq_bar:.3f} ± {se_Wq:.3f}   (exact {Wq_exact:.4f})")
print(f"  P(Wq=0) = {p0_hat:.4f} ± {se_p0:.4f} (exact {1 - RHO:.4f})")
print(f"  P(W>10) = {p10_hat:.4f} ± {se_p10:.4f} (exact e^-2 = {p10_exact:.4f})")
print(f"  std(W) = {Ww.std():.3f} ~ mean(W) = {Wbar:.3f}  (exponential: std = mean)")

In [ ]:
x = np.linspace(0, Ww.max(), 400)
ax = plot_hist_vs_pdf(Ww, x, (MU - LAM) * np.exp(-(MU - LAM) * x), bins=60,
                      label_pdf="Exp(mu - lambda) pdf")
ax.set_xlabel("sojourn time W"); ax.set_ylabel("density")
ax.set_title("Fig 3: Sojourn time W of M/M/1 customers (after warm-up) vs Exp(mu - lambda)")
ax.set_xlim(0, 30)
plt.show()

히스토그램이 곡선 $0.2e^{-0.2w}$를 따르면 정리 4가 확인된 것이다: 지수분포는 최빈값이 0이고(기다리지 않고 짧은 서비스를 받은 손님), 표준편차가 평균과 같다.

### 4.5 리틀의 법칙: 면적 항등식과 $\hat L$ (단계 3b)

전체 $[0,T]$, $T=D_n$에서 $N(t)$ 아래 면적을 계산한다. 시각 0과 $T$에 시스템이 비어 있으므로 정리 2가 **정확히** 성립해야 한다: `area == W.sum()`. 그 뒤 $\hat L=\text{area}/T$를 $\rho/(1-\rho)$와 비교한다 — 이것은 항등식이 아니라 진짜 검증이다(트랩 3).

In [ ]:
T = D[-1]
area = np.sum(N[:-1] * np.diff(t_ev))          # 세로로 세기: sum N(t) dt
print(f"area under N(t) = {area:.6f},  sum of W_i = {W.sum():.6f}  -> identical? {np.isclose(area, W.sum())}")
assert np.isclose(area, W.sum())               # 정리 2 (표본경로 항등식, 정확)

L_hat, lam_hat = area / T, n / T
print(f"L_hat = area/T = {L_hat:.4f},  lam_hat = n/T = {lam_hat:.4f},  lam_hat * mean(W) = {lam_hat * W.mean():.4f}")
# 워밍업 이후 구간 [A_warm, T] 의 시간평균 (참고: 초기 과도의 영향)
mask = t_ev[:-1] >= A[warm]
L_hat_warm = np.sum(N[:-1][mask] * np.diff(t_ev)[mask]) / (T - A[warm])
print(f"time average of N(t) after warm-up = {L_hat_warm:.4f}   (exact L = {L_exact:.4f})")
se_L = lam_hat * se_W                           # L_hat = lam_hat * W bar 이므로 SE 도 같은 비율
print(f"L_hat = {L_hat:.3f} ± {se_L:.3f}")

### 4.6 PASTA: 도착이 보는 손님 수 (단계 4)

손님 $i$가 도착 직전에 보는 손님 수는 (앞선 도착 수) − (그때까지의 퇴장 수). FIFO라 $D$가 정렬되어 있으므로 `searchsorted`로 센다. 경험 pmf를 $(1-\rho)\rho^n$과 겹치고 평균을 $L$과 비교한다. '$W_q=0$'과 '도착 시 빈 시스템'은 같은 사건이므로 두 비율은 정확히 같아야 한다.

In [ ]:
seen = np.arange(n) - np.searchsorted(D, A, side="right")   # N(A_i^-)
seen_w = seen[warm:]
seen_mean, se_seen = seen_w.mean(), batch_means_se(seen_w.astype(float), 20)
support = np.arange(0, seen_w.max() + 1)
ax = plot_hist_vs_pmf(seen_w, support, (1 - RHO) * RHO**support, label_pmf="(1-rho) rho^n")
ax.set_xlim(-0.5, 30.5); ax.set_xlabel("number in system seen by an arrival")
ax.set_title("Fig 4: Number seen by arrivals (PASTA) vs stationary geometric pmf")
plt.show()
print(f"mean seen by arrivals = {seen_mean:.3f} ± {se_seen:.3f}  (L = {L_exact:.4f})")
print(f"P(arrival sees empty) = {(seen_w == 0).mean():.4f} == P(Wq = 0) = {p0_hat:.4f}: {(seen_w == 0).mean() == p0_hat}")

### 4.7 $\rho\to1$ 폭발 (단계 5)

$\mu=1$을 고정하고 $\rho\in\{0.5,0.7,0.8,0.9,0.95\}$에서 각각 손님을 시뮬레이션해 $\bar W$와 배치 SE를 $1/(\mu-\lambda)$ 곡선 위에 얹는다. 손님 수는 같은데 $\rho$가 커질수록 오차막대가 커지고, $\rho=0.95$에서는 워밍업(10%)이 완화 시간 $1/(1-\sqrt\rho)^2\approx1500$ 시간 단위에 못 미쳐 아래로 치우칠 수 있다. 여기서는 관찰만 한다(assert 없음).

In [ ]:
n5 = scale(50_000)
rhos = np.array([0.5, 0.7, 0.8, 0.9, 0.95])
means, ses = [], []
for r in rhos:
    A5, D5 = mm1_customers(r * MU, MU, n5, rng)
    W5 = (D5 - A5)[n5 // 10:]
    means.append(W5.mean()); ses.append(batch_means_se(W5, 20))
    print(f"rho={r:.2f}: W bar = {W5.mean():7.3f} ± {ses[-1]:.3f}   exact {1/(MU - r*MU):6.2f}   "
          f"relaxation time ~ {1/(1 - np.sqrt(r))**2:7.1f}")
fig, ax = plt.subplots()
ax.plot(rho_grid, 1 / (MU - rho_grid * MU), label="1 / (mu - lambda)")
ax.errorbar(rhos, means, yerr=np.array(ses) * 1.96, fmt="o", color="C3", capsize=4,
            label=f"simulation, n={n5} per rho (95% batch-means CI)")
ax.set_yscale("log"); ax.set_xlabel("rho"); ax.set_ylabel("mean sojourn time W")
ax.set_title("Fig 5: Mean sojourn time vs rho with batch-means error bars")
ax.legend()
plt.show()

### 4.8 SE 정직성: i.i.d. 공식 vs 배치 평균 (단계 6)

4.4의 체류시간에 대해 i.i.d. 공식 `std/sqrt(n)` (= `mc_estimate(...).se`)과 배치 평균 SE의 비율을 본다. 이론상 $\rho=0.8$에서 약 10배. 배치 수를 바꿔도 배치 SE가 안정적인지도 확인한다.

In [ ]:
se_iid = mc_estimate(Ww).se
se_batch = batch_means_se(Ww, 20)
print(f"iid SE = {se_iid:.4f},  batch-means SE (b=20) = {se_batch:.4f},  ratio = {se_batch / se_iid:.1f}")
print(f"effective sample size ~ n / ratio^2 = {(n - warm) / (se_batch / se_iid)**2:.0f} of {n - warm}")
print("| batches b | batch SE of W bar |\n|---|---|")
for b in (10, 20, 50):
    print(f"| {b} | {batch_means_se(Ww, b):.4f} |")
assert se_batch / se_iid > 3   # 자기상관이 iid SE 를 심하게 낙관적으로 만든다

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
rows = [
    {"name": "W: mean sojourn (step 2)", "predicted": predictions["W_mm1"], "estimate": Wbar, "se": se_W, "exact": W_exact},
    {"name": "Wq: mean wait (step 2)", "predicted": predictions["Wq_mm1"], "estimate": Wq_bar, "se": se_Wq, "exact": Wq_exact},
    {"name": "P(no wait) = P(Wq=0) (step 2)", "predicted": predictions["p_no_wait"], "estimate": p0_hat, "se": se_p0, "exact": 1 - RHO},
    {"name": "P(W > 10) (step 2)", "predicted": predictions["p_sojourn_gt_10"], "estimate": p10_hat, "se": se_p10, "exact": p10_exact},
    {"name": "L: time average of N(t) (step 3)", "predicted": predictions["L_mm1"], "estimate": L_hat, "se": se_L, "exact": L_exact},
    {"name": "L seen by arrivals, PASTA (step 4)", "predicted": None, "estimate": seen_mean, "se": se_seen, "exact": L_exact},
    {"name": "L at rho=0.95 [mmc_metrics vs rho/(1-rho)]", "predicted": predictions["L_rho_095"], "estimate": mmc_metrics(0.95, 1.0, 1)["L"], "exact": 0.95 / 0.05},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **W**: 정리 4($\mathrm{Exp}(\mu-\lambda)$)나 Lindley 재귀 구현이 틀렸거나, SE가 과소(배치 수 부족)하거나 워밍업이 부족한 것.
- **Wq**: $W$가 맞는데 $W_q$만 어긋나면 서비스 시작 시각 `start`의 계산($\max(A_i,D_{i-1})$)이 잘못된 것.
- **P(no wait)**: PASTA(정리 3) 또는 $\pi_0=1-\rho$가 어긋난 것 — 도착이 시간평균과 다른 것을 본다는 뜻.
- **P(W > 10)**: 평균은 맞는데 꼬리가 어긋나면 체류시간이 지수가 아니라는 뜻(예: FIFO가 아닌 규칙).
- **L (time average)**: 리틀의 항등식은 정확하므로, 이 행이 어긋나면 $W$ 행도 같이 어긋나야 한다($\hat L=\hat\lambda\bar W$). 혼자 어긋나면 `number_in_system`의 면적 계산을 의심한다.
- **L seen by arrivals**: PASTA의 평균 버전. 위 L 행과 다르게 어긋나면 `seen` 계산(정렬·`side`)을 의심한다.
- **L at rho=0.95**: 두 닫힌 형식(Erlang C의 $c=1$ vs 정리 1)의 대수적 일치 — 어긋나면 `mmc_metrics`의 버그다.

In [ ]:
assert_close(Wbar, W_exact, se=se_W, k=4, name="W_mm1")
assert_close(Wq_bar, Wq_exact, se=se_Wq, k=4, name="Wq_mm1")
assert_close(p0_hat, 1 - RHO, se=se_p0, k=4, name="p_no_wait")
assert_close(p10_hat, p10_exact, se=se_p10, k=4, name="p_sojourn_gt_10")
assert_close(L_hat, L_exact, se=se_L, k=4, name="L_mm1")
assert_close(seen_mean, L_exact, se=se_seen, k=4, name="L_seen_pasta")
print("all Monte Carlo checks passed (4 SE, batch-means SE)")

## 6. 연습문제

### E1 계산

손으로: 창구 하나에 시간당 3명이 푸아송으로 도착하고 서비스는 평균 15분의 지수분포다. $\rho, L, W, L_q, W_q$와 $P(N\ge5)$를 구하라. 서비스 평균을 12분으로 줄이면 $W$는 얼마나 줄어드는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

시간 단위로 $\lambda=3$, $\mu=4$ (15분 = 1/4시간): $\rho=3/4$. 정리 1·4로
$L=\rho/(1-\rho)=3$, $W=1/(\mu-\lambda)=1$시간, $W_q=W-1/\mu=0.75$시간, $L_q=\lambda W_q=2.25$ (또는 $L-\rho$).
$P(N\ge5)=\sum_{n\ge5}(1-\rho)\rho^n=\rho^5=0.2373$.

서비스 평균 12분이면 $\mu=5$: $W=1/(5-3)=0.5$시간. 서비스 시간을 20% 줄였는데 체류시간은 **50%** 줄었다 — $\rho$가 0.75에서 0.6으로 내려가며 $1/(1-\rho)$ 배율이 4에서 2.5로 떨어지기 때문이다.

```python
lam, mu = 3.0, 4.0
rho = lam / mu
m1 = mmc_metrics(lam, mu, 1)
print(rho, m1["L"], m1["W"], m1["Lq"], m1["Wq"], rho**5)          # 0.75 3.0 1.0 2.25 0.75 0.2373
print(mmc_metrics(lam, 5.0, 1)["W"])                               # 0.5
```

</details>

### E2 유도 후 시뮬레이션 검증

$P(W_q>t)=\rho\,e^{-(\mu-\lambda)t}$를 PASTA와 기하합으로 유도하고, $\rho=0.8,\mu=1,t=2$에서 4.4의 시뮬레이션 자료로 `(Wq > 2).mean()`(워밍업 후, 배치 SE)을 닫힌 형식과 비교하라(`assert_close`, k=4).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** 정리 4 증명의 대기 버전이다. PASTA로 도착 손님이 앞에 $n$명을 볼 확률은 $(1-\rho)\rho^n$. $n=0$이면 $W_q=0$; $n\ge1$이면 무기억성으로 $W_q$는 $n$개의 i.i.d. $\mathrm{Exp}(\mu)$ 합, $\mathrm{Erlang}(n,\mu)$. 라플라스 변환:
$$E[e^{-sW_q}\mathbf 1\{W_q>0\}]=\sum_{n\ge1}(1-\rho)\rho^n\Big(\frac{\mu}{\mu+s}\Big)^n=(1-\rho)\frac{\rho\mu/(\mu+s)}{1-\rho\mu/(\mu+s)}=\rho\cdot\frac{\mu-\lambda}{\mu-\lambda+s}.$$
즉 $W_q\mid W_q>0\sim\mathrm{Exp}(\mu-\lambda)$이고 $P(W_q>0)=\rho$, 따라서 $P(W_q>t)=\rho\,e^{-(\mu-\lambda)t}$. $t=2$: $0.8e^{-0.4}=0.5363$.

```python
t0 = 2.0
p2_exact = RHO * np.exp(-(MU - LAM) * t0)
ind = (Wqw > t0).astype(float)
p2_hat, se_p2 = ind.mean(), batch_means_se(ind, 20)
print(f"P(Wq > 2): sim {p2_hat:.4f} ± {se_p2:.4f}, exact {p2_exact:.4f}")   # 0.5363
assert_close(p2_hat, p2_exact, se=se_p2, k=4, name="P(Wq>2)")
```

</details>

### E3 가정을 깨보기

도착 간격을 $\mathrm{Exp}(\lambda)$ 대신 **상수** $1/\lambda=1.25$로 바꾼 D/M/1($\mu=1$, $\rho=0.8$)을 `mm1_customers`를 수정해 시뮬레이션하라. (a) 리틀의 법칙 $\hat L=\hat\lambda\bar W$는 여전히 성립하는가? (b) 도착이 보는 빈 시스템의 비율은 시간평균 $\pi_0=1-\rho=0.2$와 같은가(PASTA)? (c) $\bar W$는 M/M/1의 5와 어떻게 다른가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) **성립한다.** 정리 2는 표본경로 항등식이라 도착 분포와 무관하다: 시뮬레이션에서 `area / T`와 `lam_hat * W.mean()`이 부동소수점 수준에서 같다.

(b) **실패한다.** G/M/1 이론에서 도착이 보는 $N$은 $\mathrm{Geom}(\sigma)$, $\sigma$는 $\sigma=E[e^{-\mu(1-\sigma)\,\tau}]$($\tau$ = 도착 간격)의 $(0,1)$ 해다. D/M/1이면 $\sigma=e^{-\mu(1-\sigma)/\lambda}=e^{-1.25(1-\sigma)}$, 수치해 $\sigma=0.62863$. 도착이 보는 빈 비율은 $1-\sigma\approx0.371$이지만, **시간평균** 빈 비율은 여전히 $1-\rho=0.2$다(서버 이용률은 리틀로 $\lambda\cdot\frac1\mu=\rho$; 시뮬레이션의 `N(t)==0` 시간 비율로 확인). 규칙적 도착은 앞 손님이 떠날 시간을 '벌어 주므로' 도착이 시간평균보다 빈 시스템을 더 자주 본다.

(c) $W_q=\sigma/(\mu(1-\sigma))=1.693$, $W=W_q+1/\mu=2.693<5$. 도착의 변동성이 사라져 대기가 크게 준다 (Kingman 근사 $W_q\approx\frac{\rho}{1-\rho}\cdot\frac{c_a^2+c_s^2}{2}\cdot\frac1\mu=4\cdot\frac{0+1}{2}=2$와 같은 방향). 대기가 줄었으니 리틀로 $L$도 $0.8\times2.693\approx2.15$로 준다.

```python
from scipy.optimize import brentq

def dm1_customers(lam, mu, n_customers, rng):
    """D/M/1: 도착 간격 상수 1/lam, 서비스 Exp(mu), FIFO."""
    A = np.arange(1, n_customers + 1) / lam                 # 결정론적 도착
    S = rng.exponential(1.0 / mu, n_customers)
    D = np.empty(n_customers); prev = 0.0
    for i in range(n_customers):
        prev = max(A[i], prev) + S[i]; D[i] = prev
    return A, D

A3, D3 = dm1_customers(LAM, MU, n, rng)
W3 = D3 - A3; warm3 = n // 10
t3, N3 = number_in_system(A3, D3); T3 = D3[-1]
area3 = np.sum(N3[:-1] * np.diff(t3))
print("(a) L_hat =", area3 / T3, " lam_hat * W bar =", n / T3 * W3.mean())      # 같다 (항등식)
seen3 = (np.arange(n) - np.searchsorted(D3, A3, side="right"))[warm3:]
p_empty_time = np.sum(np.diff(t3)[N3[:-1] == 0]) / T3
sigma = brentq(lambda s: s - np.exp(-MU * (1 - s) / LAM), 1e-9, 1 - 1e-9)
print(f"(b) arrivals see empty: {(seen3 == 0).mean():.4f} ± {batch_means_se((seen3 == 0).astype(float)):.4f}"
      f"  (G/M/1 theory 1-sigma = {1 - sigma:.4f});  time-average empty = {p_empty_time:.4f} (1-rho = 0.2)")
print(f"(c) W bar = {W3[warm3:].mean():.3f} ± {batch_means_se(W3[warm3:]):.3f}"
      f"  (theory {sigma / (MU * (1 - sigma)) + 1 / MU:.3f}; M/M/1 = 5)")
```

</details>

## 7. 정리

1. M/M/1($\rho<1$): $\pi_n=(1-\rho)\rho^n$, $L=\rho/(1-\rho)$, $L_q=\rho^2/(1-\rho)$, $W=1/(\mu-\lambda)$, $W_q=\rho/(\mu-\lambda)$ — $\rho\to1$에서 폭발한다.
2. 리틀의 법칙 $L=\lambda W$는 계단함수 아래 면적을 두 방향으로 센 회계 항등식이라 분포·서비스 규칙에 무관하다. 시뮬레이션에서 자동으로 맞으니, 검증 대상은 닫힌 형식 쪽이다.
3. PASTA: 푸아송 도착이 보는 상태 분포 = 시간평균 분포. 그래서 $P(\text{기다리지 않음})=1-\rho$. 푸아송이 아니면(D/M/1) 실패한다.
4. FIFO M/M/1의 체류시간은 정확히 $\mathrm{Exp}(\mu-\lambda)$ — 기하 개수의 지수 합은 지수.
5. 손님 단위 자료는 자기상관이 강하다: SE는 배치 평균으로, 워밍업은 버리고, $\rho\to1$일수록 둘 다 커진다.

**trap 카드**
- Q: 손님 20만 명의 체류시간을 시뮬레이션했다. $\bar W$의 표준오차를 `W.std()/sqrt(200000)`으로 계산해도 되는가?
- A: 안 된다. 연속한 손님의 체류시간은 강하게 상관되어($\rho=0.8$에서 유효 표본 수 ≈ 1/100) 그 SE는 약 10배 과소평가된다. 20개 배치 평균의 표준편차/$\sqrt{20}$ (또는 독립 반복)을 써야 4 SE 판정이 의미 있다.

**다음 노트북**: 03e — 서버가 여러 대이거나 대기실이 유한하면 곱 형태가 Erlang C/B 공식을 주고, '빠른 서버 하나 vs 느린 서버 둘' 같은 설계 질문에 $W$와 $W_q$가 다른 답을 준다.

log/progress.md 한 줄 템플릿:
`- [ ] 03d M/M/1 대기행렬과 리틀의 법칙 — 날짜: ____ / 예측 적중: __/6 / E1 __ E2 __ E3 __ / 메모: ____`